In [1]:
from pennylane.fermi import FermiSentence, from_string
import pennylane as qml
from pennylane import numpy as np
import time
qml.about()

Name: pennylane
Version: 0.43.1
Summary: PennyLane is a cross-platform Python library for quantum computing, quantum machine learning, and quantum chemistry. Train a quantum computer the same way as a neural network.
Home-page: 
Author: 
Author-email: 
License-Expression: Apache-2.0
Location: /home/lzs/.conda/envs/lzsgpu/lib/python3.12/site-packages
Requires: appdirs, autograd, autoray, cachetools, diastatic-malt, networkx, numpy, packaging, pennylane-lightning, requests, rustworkx, scipy, tomlkit, typing_extensions
Required-by: pennylane_lightning, pennylane_lightning_gpu

Platform info:           Linux-6.11.0-26-generic-x86_64-with-glibc2.39
Python version:          3.12.12
Numpy version:           2.3.4
Scipy version:           1.16.2
JAX version:             0.6.2
Installed devices:
- default.clifford (pennylane-0.43.1)
- default.gaussian (pennylane-0.43.1)
- default.mixed (pennylane-0.43.1)
- default.qubit (pennylane-0.43.1)
- default.qutrit (pennylane-0.43.1)
- default.qutrit.mix

In [2]:
# 1. 导入必要库（和之前一致，必须用 scipy.sparse）
import scipy.sparse as sp

# 2. 直接写文件名（相对路径，同目录下无需加任何路径前缀）
matrix_file = "L=5 n=4.npz"  # 重点：只写文件名，不用加 /Users/... 之类的路径

# 3. 加载矩阵（一行搞定，自动恢复 CSR 格式）
H_4 = sp.load_npz(matrix_file)

# 4. 验证加载成功（可选，快速确认没出错）
print("✅ 矩阵调用成功！")
print(f"矩阵格式：{H_4.format}")  # 输出 'csr'（和保存时一致）
print(f"矩阵形状：{H_4.shape}")    # 输出原矩阵的行数×列数
print(f"非零元素个数：{H_4.nnz}")  # 输出非零元素数量

from scipy.sparse.linalg import eigsh
from scipy.linalg import eigh
H= H_4.toarray()
# 计算模最小的特征值（注意可能为复数）
min_eigval = eigsh(H, k=2, which='SA', return_eigenvectors=True,)

min =  eigh( H,eigvals_only=True)[0]
print("最小特征值:", min_eigval[0])
print(min)

✅ 矩阵调用成功！
矩阵格式：csr
矩阵形状：(2227, 2227)
非零元素个数：81115
最小特征值: [-35.68407846 -15.41150228]
-35.684078464608774


In [3]:

def get_Hami(H):
    """
    将任意维度的哈密顿量矩阵扩展为 2^Nq 维度（量子比特系统希尔伯特空间维度）

    参数:
        H: 原哈密顿量矩阵（quspin 哈密顿量或 numpy 数组，维度为 d×d）
    返回:
        Hami: 扩展后的哈密顿量矩阵（维度 l×l，l=2^Nq）
        Nq: 所需量子比特数（满足 2^Nq ≥ d 的最小整数）
    说明:
        量子比特系统的希尔伯特空间维度必为 2 的幂次，原玻色子基矢空间维度 Ns 可能非 2^Nq，
        故通过补零扩展至最近的 2^Nq 维度，确保与量子计算框架兼容
    """
    # 若输入为 quspin 哈密顿量，先转换为 numpy 数组（优先保留稀疏性，此处暂转 dense 便于理解）
    if hasattr(H, 'toarray'):
        H_dense = H.toarray()
    else:
        H_dense = np.asarray(H)

    d = H_dense.shape[0]  # 原哈密顿量维度（即玻色子基矢空间维度 Ns）
    Nq = int(np.ceil(np.log2(d)))  # 满足 2^Nq ≥ d 的最小量子比特数
    l = 2 ** Nq  # 扩展后的维度（量子比特系统维度）

    # 初始化扩展矩阵（补零）
    Hami = np.zeros((l, l), dtype=H_dense.dtype)
    Hami[:d, :d] = H_dense  # 原矩阵嵌入扩展矩阵的左上角，其余位置补零

    return Hami, Nq

Hami ,n_qubits  = get_Hami(H)

In [4]:
import pennylane.optimize as opt
print([x for x in dir(opt) if x.endswith('Optimizer')])


['AdagradOptimizer', 'AdamOptimizer', 'AdaptiveOptimizer', 'GradientDescentOptimizer', 'MomentumOptimizer', 'MomentumQNGOptimizer', 'NesterovMomentumOptimizer', 'QNGOptimizer', 'QNSPSAOptimizer', 'RMSPropOptimizer', 'RiemannianGradientOptimizer', 'RotoselectOptimizer', 'RotosolveOptimizer', 'SPSAOptimizer', 'ShotAdaptiveOptimizer']


In [17]:
import pennylane as qml
from pennylane import numpy as np
from pennylane.optimize import AdamOptimizer,RotosolveOptimizer

##############################
# 1. 问题参数：比特数、HEA 深度
##############################
n_qubits = 12
N = n_qubits             # 任意 N 比特
depth = 20         # HEA 层数
steps = 400        # VQE 迭代步数
lr = 0.05          # Adam 学习率

##############################
# 2. 定义哈密顿量（Transverse-Field Ising）
##############################
         # hx

##############################
# 3. 构造 Hardware-Efficient Ansatz
#    每层：单比特旋转 RX-RY-RX + CNOT 纠缠
##############################
def hea_layer(params):
    """单个 HEA 层：params 形状 (3, N)"""
    for i in range(N):
        qml.RX(params[0, i], wires=i)
        qml.RY(params[1, i], wires=i)
        qml.RX(params[2, i], wires=i)
    # 线性邻接 CNOT 纠缠
    for i in range(N - 1):
        qml.CNOT(wires=[i, i + 1])

def ansatz(params):
    """depth 层 HEA"""
    params = params.reshape((depth, 3, N))
    for d in range(depth):
        hea_layer(params[d])

##############################
# 4. 定义 QNode：返回能量期望值
##############################
dev = qml.device("lightning.gpu", wires=N)

@qml.qnode(dev, interface="autograd")
def energy_fn(params):
    ansatz(params)
    return qml.expval(qml.Hermitian(Hami, wires=range(n_qubits)))

##############################
# 5. 初始化参数并运行 VQE
##############################
params = np.zeros(size=(depth, 3, N), requires_grad=True)

opt = qml.optimize.NesterovMomentumOptimizer(0.01, 0.9)

print("开始 VQE 优化...")
for step in range(steps + 1):
    params, loss = opt.step_and_cost(energy_fn, params)
    if step % 10 == 0:
        print(f"Step {step:3d} | energy = {loss:.8f}")

print("\n最终基态能量估计:", energy_fn(params))

##############################
# 6. 可选：打印最终线路
##############################
# 在 PennyLane 0.43 里，只要这样即可：
# print(qml.draw(energy_fn)(params))




开始 VQE 优化...


/home/lzs/.conda/envs/lzsgpu/lib/python3.12/site-packages/pennylane/devices/preprocess.py:283: UserWarning: Differentiating with respect to the input parameters of Hermitian is not supported with the adjoint differentiation method. Gradients are computed only with regards to the trainable parameters of the circuit.

 Mark the parameters of the measured observables as non-trainable to silence this warning.
  warnings.warn(


Step   0 | energy = 311.07797792
Step  10 | energy = 264.31012816
Step  20 | energy = 272.36662715
Step  30 | energy = 280.58802638
Step  40 | energy = 294.92442798
Step  50 | energy = 304.42328700
Step  60 | energy = 284.90215206
Step  70 | energy = 286.26692309
Step  80 | energy = 292.10767390
Step  90 | energy = 260.16791365
Step 100 | energy = 272.95459830
Step 110 | energy = 284.47446637
Step 120 | energy = 276.04946094
Step 130 | energy = 309.95414366
Step 140 | energy = 277.00020747
Step 150 | energy = 274.79510677
Step 160 | energy = 291.29318571
Step 170 | energy = 280.92646710
Step 180 | energy = 271.97875062
Step 190 | energy = 297.35735631
Step 200 | energy = 287.18578081
Step 210 | energy = 298.57487040
Step 220 | energy = 290.32397587
Step 230 | energy = 298.20748867
Step 240 | energy = 285.42764851
Step 250 | energy = 265.77618052
Step 260 | energy = 293.19241935
Step 270 | energy = 299.75662372
Step 280 | energy = 284.82650667
Step 290 | energy = 316.17769406
Step 300 |

In [ ]:
import pennylane as qml
from pennylane import numpy as np
from pennylane.optimize import AdamOptimizer, RotosolveOptimizer

##############################
# 1. 问题参数：比特数、HEA 深度
##############################
n_qubits = 12
N = n_qubits             # 任意 N 比特
depth = 20               # HEA 层数
steps = 400              # VQE 迭代步数
lr = 0.05                # Adam 学习率（Rotosolve 不用）

##############################
# 2. 定义哈密顿量（Transverse-Field Ising）
##############################
# 这里你原来怎么定义 Hami 就怎么来
# Hami = ...

##############################
# 3. 构造 Hardware-Efficient Ansatz
##############################
def hea_layer(params):
    """单个 HEA 层：params 形状 (3, N)"""
    for i in range(N):
        qml.RX(params[0, i], wires=i)
        qml.RY(params[1, i], wires=i)
        qml.RX(params[2, i], wires=i)
    # 线性邻接 CNOT 纠缠
    for i in range(N - 1):
        qml.CNOT(wires=[i, i + 1])

def ansatz(params):
    """depth 层 HEA"""
    params = params.reshape((depth, 3, N))
    for d in range(depth):
        hea_layer(params[d])

##############################
# 4. 定义 QNode：返回能量期望值
##############################
dev = qml.device("lightning.gpu", wires=N)

@qml.qnode(dev, interface="autograd")
def energy_fn(params):
    ansatz(params)
    return qml.expval(qml.Hermitian(Hami, wires=range(n_qubits)))

##############################
# 5. 初始化参数
##############################
params = np.random.uniform(0, 2 * np.pi, size=(depth, 3, N), requires_grad=True)

##############################
# 5.1 为 Rotosolve 设置频率信息（关键！！）
##############################
# params 的名字就是 energy_fn 的参数名 "params"
# 形状 (depth, 3, N)，每个标量参数都只在一个 RX/RY/RX 旋转里，
# 对应的频率个数 = 1
nums_frequency = {
    "params": { (d, k, i): 1
                for d in range(depth)
                for k in range(3)
                for i in range(N) }
}

##############################
# 5.2 选择 Rotosolve 优化器
##############################
opt = qml.RotosolveOptimizer()

print("开始 VQE 优化（Rotosolve）...")
for step in range(steps + 1):
    params, loss = opt.step_and_cost(
        energy_fn,
        params,
        nums_frequency=nums_frequency,   # ⭐ 一定要传这一项
    )
    if step % 10 == 0:
        print(f"Step {step:3d} | energy = {loss:.8f}")

print("\n最终基态能量估计:", energy_fn(params))

##############################
# 6. 可选：打印最终线路
##############################
# print(qml.draw(energy_fn)(params))
